# LLM Integration

This notebook demonstrates how to use LLM-generated environments with different backends: Mock LLM, OpenAI, and Ollama.

**LLM backends:**
- **MockLLMClient**: Built-in mock for testing (no API key needed)
- **OpenAILLMClient**: OpenAI ChatGPT (requires `openai` package and API key)
- **OllamaLLMClient**: Local Ollama API (requires running Ollama server)
- **AnthropicLLMClient**: Anthropic Claude (requires `anthropic` package and API key)

In [ ]:
import sys
sys.path.insert(0, '../..')

from envgen.environments.base import EnvironmentConfig
from envgen.utils.types import DifficultyLevel

## 1. Mock LLM Client

The built-in mock LLM generates deterministic environments for testing without any API keys.

In [ ]:
from envgen.llm import MockLLMClient
from envgen.llm.generator import EnvironmentGenerator

# Create mock LLM client
llm = MockLLMClient()
generator = EnvironmentGenerator(llm_client=llm)

# Generate an environment
config = generator.generate(
    task_description="A navigation task where the agent collects items",
    difficulty=DifficultyLevel.EASY,
    target_skills=["navigation", "planning"],
)

print(f"Generated environment: {config.name}")
print(f"Difficulty: {config.difficulty}")
print(f"Parameters: {config.parameters}")
print(f"Target skills: {config.target_skills}")

## 2. Environment Generation with Feedback Loop

Demonstrate how the LLM adapts environments based on evaluation feedback.

In [ ]:
from envgen.llm import MockLLMClient
from envgen.llm.generator import EnvironmentGenerator
from envgen.evaluators import SkillEvaluator
from envgen.utils.types import SkillScore, EvaluationResult

llm = MockLLMClient()
generator = EnvironmentGenerator(llm_client=llm)

# Generate initial environment
config1 = generator.generate(
    task_description="A puzzle navigation task",
    difficulty=DifficultyLevel.EASY,
    target_skills=["navigation", "planning"],
)
print(f"Cycle 1 - {config1.name} (difficulty: {config1.difficulty})")
print(f"  Parameters: {config1.parameters}")

# Simulate evaluation results showing weak planning
weak_skills = ["planning"]
strong_skills = ["navigation"]

# Adapt environment based on feedback
config2 = generator.adapt(
    previous_config=config1,
    evaluation_result=EvaluationResult(
        agent_id="test_agent",
        skill_scores=[
            SkillScore(skill_name="navigation", score=0.8),
            SkillScore(skill_name="planning", score=0.3),
            SkillScore(skill_name="reasoning", score=0.5),
            SkillScore(skill_name="memory", score=0.6),
            SkillScore(skill_name="efficiency", score=0.4),
        ],
        overall_score=0.52,
        weak_skills=weak_skills,
        strong_skills=strong_skills,
    ),
    cycle=1,
)
print(f"\nCycle 2 - {config2.name} (difficulty: {config2.difficulty})")
print(f"  Parameters: {config2.parameters}")
print(f"  Adapted to improve: {weak_skills}")

## 3. OpenAI LLM Client (Optional)

Use OpenAI's ChatGPT for environment generation. Requires `openai` package and `OPENAI_API_KEY` environment variable.

In [ ]:
try:
    from envgen.llm import OpenAILLMClient
    import os
    
    # Check if API key is set
    if os.environ.get("OPENAI_API_KEY"):
        llm = OpenAILLMClient(model="gpt-3.5-turbo")
        generator = EnvironmentGenerator(llm_client=llm)
        
        config = generator.generate(
            task_description="A challenging maze navigation task",
            difficulty=DifficultyLevel.MEDIUM,
            target_skills=["navigation", "reasoning"],
        )
        print(f"OpenAI generated: {config.name}")
        print(f"Parameters: {config.parameters}")
    else:
        print("OPENAI_API_KEY not set. Set it to use OpenAI LLM client.")
        print("export OPENAI_API_KEY='your-api-key-here'")
except ImportError:
    print("OpenAI package not installed. Install with: pip install openai")

## 4. Ollama LLM Client (Optional)

Use a local Ollama server for environment generation. No API key needed, but requires a running Ollama server.

In [ ]:
try:
    from envgen.llm import OllamaLLMClient
    
    # Check if Ollama is available
    llm = OllamaLLMClient(model="llama3.2")
    
    if llm.is_available():
        generator = EnvironmentGenerator(llm_client=llm)
        
        config = generator.generate(
            task_description="A resource management task",
            difficulty=DifficultyLevel.EASY,
            target_skills=["reasoning", "planning"],
        )
        print(f"Ollama generated: {config.name}")
        print(f"Parameters: {config.parameters}")
    else:
        print("Ollama server not available. Start Ollama with: ollama serve")
        print("Make sure the model is pulled: ollama pull llama3.2")
except ImportError:
    print("Ollama client not available")

## 5. Full Training Pipeline with LLM

Run the complete training pipeline with LLM-generated environments.

In [ ]:
from envgen.pipeline import TrainingPipeline
from envgen.pipeline.trainer import PipelineConfig
from envgen.llm import MockLLMClient

# Configure the pipeline
config = PipelineConfig(
    task_description="Navigate a grid world and collect items",
    num_cycles=3,
    episodes_per_cycle=10,
    agent_config={"type": "ppo", "learning_rate": 0.1, "epsilon": 0.2},
    environment_config={"type": "puzzle"},
    llm_config={"type": "mock"},
)

# Run the pipeline
pipeline = TrainingPipeline(config=config)
results = pipeline.train()

print("\nTraining complete!")
print(f"Final skill scores:")
for skill, score in results.final_skill_scores.items():
    print(f"  {skill}: {score:.3f}")

## Summary

EnvGen-LLM supports multiple LLM backends for environment generation:

| Backend | Package | API Key | Use Case |
|---------|---------|---------|----------|
| MockLLMClient | None | No | Testing and development |
| OpenAILLMClient | `openai` | Yes | Production with ChatGPT |
| OllamaLLMClient | None | No | Local LLM (no internet) |
| AnthropicLLMClient | `anthropic` | Yes | Production with Claude |

The `EnvironmentGenerator` provides a unified interface across all backends, making it easy to switch between them.